# NB-05 — LangChain Chains: Extraction and Confirmation

**Tutorial reference:** Part 5 — Layer 4 of the SmartIntake Learning Tutorial

This notebook builds two LangChain chains:
1. `extraction_chain` — takes a free-text regulatory message and returns the five structured fields
2. `confirmation_chain` — takes the validated fields and produces a formal compliance acknowledgement

---

## 1. Setup

In [ ]:
from dotenv import load_dotenv
import os
from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from pydantic import BaseModel, field_validator, ValidationError
from typing import Literal
import json

load_dotenv()

# ChatAnthropic is the LangChain wrapper around the Anthropic API.
# It exposes the same model through LangChain's standard interface,
# which allows it to be composed with prompts and parsers using the pipe syntax.
llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
    max_tokens=512
)

TEST_INPUTS = {
    "T1": (
        "PV team here. We have a serious unexpected SUSAR for study NCT-2244 "
        "and need to report to EMA within 15 days per ICH E2A."
    ),
    "T2": "Hi, we need to file a Type II variation for our cardiovascular product with the EMA.",
    "T3": (
        "CMC Regulatory here. FDA issued a 483 observation on our manufacturing site in Pune. "
        "We have 15 business days to respond."
    ),
    "T4": "Can you check what the weather is like in Mumbai today?",
    "T5": (
        "Labelling team. We need to update the SmPC for our oncology product "
        "following the latest EMA assessment report."
    ),
}

print("Setup complete.")

## 2. Extraction Prompt Template

A `ChatPromptTemplate` defines the structure of the messages sent to the model. The `{user_message}` placeholder is filled at runtime when the chain is invoked.

In [ ]:
# The extraction system prompt — same content as NB-03, formatted for LangChain.
# Note: double braces {{ }} are used to include literal braces in the template string
# without LangChain interpreting them as template placeholders.

EXTRACTION_SYSTEM = """
You are SmartIntake, a regulatory affairs intake specialist.
Extract the five regulatory intake fields from the user's message.
Return ONLY a JSON object with these exact keys:
query_type, regulation_ref, product_area, urgency, submitting_team.

Allowed values:
  query_type: complaint | submission | variation | safety_signal | label_update | inspection | general_enquiry
  regulation_ref: FDA_21CFR | EMA_CTR | ICH_E2A | ICH_Q10 | CDSCO_NDC | GxP_GMP | GxP_GCP | other
  product_area: oncology | cardiovascular | infectious_disease | cmc | clinical | labelling | general
  urgency: routine | standard | urgent | critical

RULES:
- Think step by step about the regulatory context before choosing enum values.
- Never infer urgency from tone. Use null if deadline is not explicitly stated.
- submitting_team must be a team name, never an individual's name. Use null if unclear.
- Use 'other' for regulation_ref if the framework is not clearly identifiable.
- If the input is not a regulatory query, return: {{"error": "non_regulatory_input"}}

EXAMPLE 1:
Input: "PV team here. SUSAR for Phase III trial, EMA deadline 15 days per ICH E2A."
Output: {{"query_type": "safety_signal", "regulation_ref": "ICH_E2A",
  "product_area": "clinical", "urgency": "critical", "submitting_team": "PV Team"}}

EXAMPLE 2:
Input: "We need to respond to an FDA query on our CMC section for NDA-209114."
Output: {{"query_type": "submission", "regulation_ref": "FDA_21CFR",
  "product_area": "cmc", "urgency": null, "submitting_team": null}}
"""

# ChatPromptTemplate.from_messages accepts a list of (role, content) tuples.
# The {user_message} placeholder will be substituted when we call .invoke().
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", EXTRACTION_SYSTEM),
    ("human", "{user_message}")
])

print("Extraction prompt template defined.")
print("--------------------------------")
print(extraction_prompt)

## 3. Building the Extraction Chain

The pipe syntax (`|`) connects components: the prompt formats the message, the LLM generates a response, and `StrOutputParser` extracts the text from the response object.

In [ ]:
def parse_json_output(text: str) -> dict:
    """
    Strip markdown code fences and parse the JSON from the LLM's text output.
    
    The model sometimes wraps JSON in ```json ... ``` fences even when instructed
    not to. This function handles both fenced and unfenced output.
    """
    clean = (
        text.strip()
        .removeprefix("```json")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )
    return json.loads(clean)


# Build the extraction chain.
# extraction_prompt: formats system + user message
# llm: calls the Anthropic API and returns a response object
# StrOutputParser(): extracts the .content text from the response object
extraction_chain = extraction_prompt | llm | StrOutputParser()

# Test the chain with T3 — a clean single-turn extraction
raw_output = extraction_chain.invoke({"user_message": TEST_INPUTS["T3"]})
print("Raw output from extraction chain:")
print(raw_output)

In [ ]:
print()
extracted = parse_json_output(raw_output)
print("Parsed dict:")
print(json.dumps(extracted, indent=2))

## 4. Confirmation Prompt Template

The confirmation chain takes the validated intake record and generates a formal regulatory acknowledgement. The tone must be professional — this is a compliance message, not a chat response.

In [ ]:
CONFIRMATION_SYSTEM = """
You are SmartIntake, a regulatory affairs intake specialist.
Given a validated structured intake record, generate a brief formal confirmation
message for the submitter to review before the record is saved.

Format requirements:
- Reference the query type and regulation formally (e.g. 'FDA 21 CFR inspection query')
- State the urgency tier by name (e.g. 'classified as urgent priority')
- Name the submitting team
- End with exactly: 'Please confirm or correct before this record is saved.'
- Keep it under 60 words
- Professional tone — this is a compliance acknowledgement, not a chat response
"""

# The {intake_record} placeholder will receive a JSON string of the validated fields.
confirmation_prompt = ChatPromptTemplate.from_messages([
    ("system", CONFIRMATION_SYSTEM),
    ("human", "Intake record: {intake_record}")
])

# Build the confirmation chain using the same pipe pattern
confirmation_chain = confirmation_prompt | llm | StrOutputParser()

print("Confirmation prompt template and chain defined.")

## 5. Running the Two-Chain Flow

This is the core of SmartIntake's LangChain layer: extract, validate, then confirm.

In [ ]:
def run_extraction_and_confirmation(user_message: str) -> tuple:
    # ===== Step 1: EXTRACT =====
    # Send the user's free-text message through the extraction chain.
    # The chain does:
    #   1. extraction_prompt fills {user_message} into the system + human message template
    #   2. llm (ChatAnthropic) sends those messages to Claude API
    #   3. StrOutputParser() extracts just the .content text from Claude's response
    # Result: 'raw' is a raw text string (e.g. '{"query_type": "inspection", ...}')
    #         OR sometimes wrapped in ```json ... ``` markdown fences
    #         OR preceded by explanatory text like 'Here is the result: {...}'
    raw = extraction_chain.invoke({"user_message": user_message})

    # Try to convert the raw text string into a Python dictionary.
    # parse_json_output handles all three output formats (fenced, unfenced, bare JSON).
    try:
        extracted = parse_json_output(raw)  # returns dict
    except json.JSONDecodeError:
        return None, f"Could not parse extraction output: {raw[:100]}"

    # Check if the LLM flagged this as NOT a regulatory query.
    # The system prompt tells Claude to return {"error": "non_regulatory_input"}
    # when the message isn't pharma-related (e.g. "What's the weather?")
    if extracted.get("error") == "non_regulatory_input":
        return None, (
            "This does not appear to be a pharmaceutical regulatory query. "
            "Please describe your compliance or regulatory obligation."
        )

    # ===== Display the extracted data =====
    print("Extracted fields:")
    print(json.dumps(extracted, indent=2))  # Pretty-print the dict as formatted JSON

    # ===== Step 2: CONFIRM =====
    # Take the extracted dict and send it through the confirmation chain.
    # The chain does:
    #   1. confirmation_prompt fills {intake_record} with the extracted dict as JSON
    #   2. llm sends this to Claude with a system prompt asking for formal acknowledgement
    #   3. StrOutputParser() extracts the text response
    confirmation = confirmation_chain.invoke(
        {"intake_record": json.dumps(extracted)}  # dict -> JSON string for template
    )
    print("Confirmation message:")
    print(confirmation)

    # Return both so the caller can use them (e.g. save to database, show to user)
    return extracted, confirmation



# Run with T3 — the clean single-turn case
print("=" * 60)
print(f"T3: {TEST_INPUTS['T3']}")
print("=" * 60)
run_extraction_and_confirmation(TEST_INPUTS["T3"])

## 6. Experiment — All Five Test Inputs

In [ ]:
# Run all five test inputs through the combined chain.
# Observe:
#   T4 (weather query) should return the non-regulatory fallback message
#   T2 and T5 will have null fields — the confirmation chain still runs but
#   the session loop (NB-06) will catch these and ask for the missing values

for key, message in TEST_INPUTS.items():
    print("\n" + "=" * 60)
    print(f"{key}: {message[:80]}..." if len(message) > 80 else f"{key}: {message}")
    print("=" * 60)
    extracted, confirmation = run_extraction_and_confirmation(message)
    if extracted is None:
        print(f"FALLBACK: {confirmation}")

---
## Summary

You have:
- Built the extraction chain using `ChatPromptTemplate | llm | StrOutputParser()`
- Built the confirmation chain with a formal regulatory acknowledgement style
- Run a complete extraction-to-confirmation flow
- Handled the non-regulatory fallback case (T4)
- Observed that T2 and T5 extract partial fields — the session loop handles those

**Next:** Open `NB-06_memory_session_loop.ipynb` to add conversation memory and the multi-turn session loop.